In [1]:
# Build the class dataset: the full Parquet (benign + attack) converted to one CSV.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-10-27_11-13-21/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_corrected_csv"

# Column order taken from the sample CSV header, so the file matches what the notebook expects
cols = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv").columns

full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
full = full.withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
full = full.select([(F.col(c).cast("string") if c in full.columns
                     else F.lit(None).cast("string")).alias(c) for c in cols])

print("Total rows:", full.count())
full.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

full.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)

26/10/27 09:56:43 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.131 instead (on interface ens33)
26/10/27 09:56:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 09:56:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/27 09:56:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

Total rows: 744052


+--------------------+------+
|label_tactic        |count |
+--------------------+------+
|none                |372194|
|Discovery           |347535|
|Reconnaissance      |23754 |
|Privilege Escalation|204   |
|Persistence         |157   |
|Lateral Movement    |100   |
|Defense Evasion     |62    |
|Initial Access      |28    |
|Exfiltration        |13    |
|Execution           |5     |
+--------------------+------+



[Stage 8:>                                                          (0 + 1) / 1]

Written to hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_corrected_csv


In [2]:
# Check the mission log against the dataset just built: which entries have labelled attack
# connections in it. Run after the build cell (uses `spark` and `full` from that cell).
import ipaddress
import pandas as pd
from IPython.display import display

mission_log_path = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/attack_log/"
                    "attack_82913-CNT4416-202608-attack_corrected.csv")
PAD = pd.Timedelta(minutes=2)          # the correlation's +/- 2 minute window

# 1. Mission log (every column as text). Line numbers match the CSV file, header = line 1.
log = (spark.read.option("header", True).option("multiline", True).option("encoding", "UTF-8")
       .csv(mission_log_path).toPandas().fillna(""))
log.insert(0, "log_line", range(2, 2 + len(log)))

# 2. Attack connections summarized per source IP, destination IP, T code and day
summary = (full.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
               .withColumn("day", F.substring("datetime", 1, 10))
               .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "day")
               .agg(F.count(F.lit(1)).alias("connections"),
                    F.min("datetime").alias("first"), F.max("datetime").alias("last"))
               .toPandas())
summary["t0"] = pd.to_datetime(summary["first"], utc=True)
summary["t1"] = pd.to_datetime(summary["last"], utc=True)
summary["matched"] = False

def network(text):
    try:
        return ipaddress.ip_network(str(text).strip(), strict=False)
    except ValueError:
        return None

def third_octet(text):
    try:
        return int(str(text).split("/")[0].split(".")[2])
    except (IndexError, ValueError):
        return None

def single_port(text):
    t = str(text).split("/")[0].strip()
    return int(t) if t.isdigit() else None

# 3. Match each entry: same T code, IPs inside the entry's IPs (or /24), time within start-2min..end+2min
rows = []
for _, e in log.iterrows():
    src, dst = network(e.src_ip), network(e.dest_ip)
    start = pd.to_datetime(e.datetime_start, utc=True, errors="coerce")
    end = pd.to_datetime(e.datetime_end, utc=True, errors="coerce")

    notes = []
    if e.mitre_attack_technique == "T0000":
        notes.append("benign window marker (T0000), not an attack")
    if pd.notna(start) and pd.notna(end) and end < start:
        notes.append("end time before start time")
    if "/" in e.src_ip:
        notes.append("source entered as a range")
    if src is not None and dst is not None and src == dst and src.num_addresses == 1:
        notes.append("source and destination are the same host (not visible on the network)")
    port = single_port(e.dest_port)
    if port is not None and port >= 32768:
        notes.append("dest port looks like an ephemeral/RHP port")
    if src is None or dst is None or pd.isna(start) or pd.isna(end):
        notes.append("could not read IP or time")

    m = summary.iloc[0:0]
    if src is not None and dst is not None and pd.notna(start) and pd.notna(end):
        m = summary[(summary.label_technique == e.mitre_attack_technique)
                    & summary.src_ip_zeek.map(lambda ip: ipaddress.ip_address(ip) in src)
                    & summary.dest_ip_zeek.map(lambda ip: ipaddress.ip_address(ip) in dst)
                    & (summary.t0 - PAD <= end) & (summary.t1 + PAD >= start)]
        summary.loc[m.index, "matched"] = True

    rows.append({
        "Log line": e.log_line, "Group": e.group_id, "T code": e.mitre_attack_technique,
        "Source IP": e.src_ip, "Dest IP": e.dest_ip, "Dest port": e.dest_port,
        "Start (UTC)": e.datetime_start, "End (UTC)": e.datetime_end,
        "Source subnet": third_octet(e.src_ip), "Target subnet": third_octet(e.dest_ip),
        "In dataset": "N/A" if e.mitre_attack_technique == "T0000" else ("Yes" if len(m) else "No"),
        "Labelled connections": int(m.connections.sum()),
        "Notes": "; ".join(notes)})
results = pd.DataFrame(rows)

# 4. Output
attacks = results[results["In dataset"] != "N/A"]
yes = lambda x: int((x["In dataset"] == "Yes").sum())
print(f"Entries in dataset: {yes(attacks)} of {len(attacks)} attack entries\n")

nets = sorted(set(attacks["Target subnet"].dropna()) | set(attacks["Source subnet"].dropna()))
tally = []
for n in nets:
    recv = attacks[attacks["Target subnet"] == n]
    self_scans = recv[recv["Source subnet"] == n]
    made = attacks[(attacks["Source subnet"] == n) & (attacks["Target subnet"] != n)]
    tally.append({"Subnet": f"143.88.{int(n)}.x",
                  "Attacks on it (in/total)": f"{yes(recv)}/{len(recv)}",
                  "of which self-scans": f"{yes(self_scans)}/{len(self_scans)}",
                  "Attacks made from it (in/total)": f"{yes(made)}/{len(made)}"})
print("Per subnet (attacks on it = DCO view; OCO = attacks on the subnet a group targeted):")
display(pd.DataFrame(tally))

problems = results[(results["Notes"] != "") & (results["In dataset"] == "No")]
print("Missing entries with an obvious problem in the log:")
display(problems[["Log line", "Group", "T code", "Source IP", "Dest IP", "Dest port", "Notes"]]
        if len(problems) else "none")

unmatched = summary[~summary["matched"]]
print(f"Labelled connection groups that matched no entry: {len(unmatched)} (should be 0)")
if len(unmatched):
    display(unmatched[["src_ip_zeek", "dest_ip_zeek", "label_technique", "first", "connections"]])

print("All entries:")
display(results)

Entries in dataset: 57 of 92 attack entries

Per subnet (attacks on it = DCO view; OCO = attacks on the subnet a group targeted):


,Subnet,Attacks on it (in/total),of which self-scans,Attacks made from it (in/total)
0,143.88.1.x,4/7,0/1,5/8
1,143.88.2.x,7/11,4/5,3/9
2,143.88.3.x,4/10,1/1,3/3
3,143.88.4.x,4/7,1/4,3/7
4,143.88.5.x,3/6,0/0,3/5
5,143.88.6.x,4/9,1/3,9/9
6,143.88.7.x,10/10,1/1,1/4
7,143.88.8.x,3/6,2/2,0/5
8,143.88.9.x,2/6,1/1,9/11
9,143.88.10.x,16/19,5/6,5/7


Missing entries with an obvious problem in the log:


,Log line,Group,T code,Source IP,Dest IP,Dest port,Notes
5,7,2,T1595,143.88.2.13,143.88.2.13,8000/tcp,source and destination are the same host (not ...
13,15,4,T1595,143.88.4.13/24,143.88.255.11,80/tcp,source entered as a range
32,34,8,T1595,143.88.8.10/24,143.88.9.10,1-1000/tcp,source entered as a range
34,36,2,T1210,143.88.2.13,143.88.3.11,34635/tcp,dest port looks like an ephemeral/RHP port
35,37,2,T1210,143.88.2.13,143.88.3.12,64291/tcp,dest port looks like an ephemeral/RHP port
42,44,8,T1190,143.88.8.13,143.88.9.11,50319/tcp,dest port looks like an ephemeral/RHP port
43,45,8,T1210,143.88.8.13,143.88.9.12,59163/tcp,dest port looks like an ephemeral/RHP port
60,62,2,T1070,143.88.2.10/24,143.88.3.12,445/tcp,source entered as a range
68,70,02,T1053,143.88.2.13,143.88.3.11,51890/tcp,dest port looks like an ephemeral/RHP port
70,72,8,T1053,143.88.8.13,143.88.9.11,38958/tcp,dest port looks like an ephemeral/RHP port


Labelled connection groups that matched no entry: 0 (should be 0)
All entries:


,Log line,Group,T code,Source IP,Dest IP,Dest port,Start (UTC),End (UTC),Source subnet,Target subnet,In dataset,Labelled connections,Notes
0,2,2,T0000,143.88.255.0/24,143.88.255.0/24,,2026-08-24T20:56:01Z,2026-08-30T20:56:06Z,255,255,N/A,0,"benign window marker (T0000), not an attack; s..."
1,3,2,T1595,143.88.2.13/24,143.88.2.1,443/tcp,2026-09-02T01:56:01Z,2026-09-02T01:56:06Z,2,2,Yes,3,source entered as a range
2,4,2,T1595,143.88.2.13,143.88.2.10,8000/tcp,2026-09-02T01:56:02Z,2026-09-02T01:57:01Z,2,2,Yes,1,
3,5,2,T1595,143.88.2.13,143.88.2.11,80/tcp,2026-09-02T01:57:30Z,2026-09-02T01:57:35Z,2,2,Yes,1,
4,6,2,T1595,143.88.2.13,143.88.2.12,80/tcp,2026-09-02T01:57:38Z,2026-09-02T01:57:40Z,2,2,Yes,1,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,90,6,T1210,143.88.6.10,143.88.7.12,445/tcp,2026-10-08T02:49:56Z,2026-10-08T02:49:56Z,6,7,Yes,22,
89,91,2,T1053,143.88.2.10,143.88.3.11,80/tcp,2026-10-07T12:35:00Z,2026-10-07T21:45:00Z,2,3,Yes,2,
90,92,2,T1021,143.88.2.10,143.88.3.12,445/tcp,2026-10-07T22:31:00Z,2026-10-07T22:38:00Z,2,3,No,0,
91,93,5,T1136,143.88.8.10,143.88.6.11,80/tcp,2026-10-05T06:17:49Z,2026-10-05T06:17:50Z,8,6,No,0,


In [3]:
results.to_csv("mission_log_checked_final.csv", index=False)

In [ ]:
# pull attack output to compare to mission logs
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

df = (spark.read.option("header", True)
      .csv("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"))
summary = (df.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
             .withColumn("day", F.substring("datetime", 1, 10))
             .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "label_tactic", "day")
             .agg(F.count(F.lit(1)).alias("connections"),
                  F.countDistinct("dest_port_zeek").alias("distinct_dest_ports"),
                  F.min("datetime").alias("first"), F.max("datetime").alias("last"))
             .orderBy("first"))
pdf = summary.toPandas()
pdf.to_csv("/home/ubuntu/CNT4416_attack_summary.csv", index=False)
print(len(pdf), "rows")
#

In [ ]:
# For every mission log entry missing from the dataset, look in the raw Zeek conn logs for traffic
# between its hosts: both directions, as logged, at +5 h (local time entered as UTC), and same day.
# Standalone: needs mission_log_checked_final.csv (from the check cell) in this folder.
import ipaddress
from datetime import timedelta
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from IPython.display import display

spark = (SparkSession.builder.master("local[*]").appName("raw_check")
         .config("spark.driver.memory", "6g").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

logs = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/nsm/zeek/logs"
PAD, SHIFT, DAY = 120, 5 * 3600, 12 * 3600

def net(text):
    try:
        return ipaddress.ip_network(str(text).strip(), strict=False)
    except ValueError:
        return None

def prefix(n):
    return ".".join(str(n.network_address).split(".")[:3]) + "."

# 1. Missing entries
results = pd.read_csv("mission_log_checked_final.csv", dtype=str).fillna("")
todo = []
for _, e in results[results["In dataset"] == "No"].iterrows():
    s, d = net(e["Source IP"]), net(e["Dest IP"])
    t0 = pd.to_datetime(e["Start (UTC)"], utc=True, errors="coerce")
    t1 = pd.to_datetime(e["End (UTC)"], utc=True, errors="coerce")
    if s is None or d is None or pd.isna(t0) or pd.isna(t1):
        continue
    t0, t1 = min(t0, t1), max(t0, t1)          # tolerate end-before-start entries
    todo.append((e["Log line"], e["Group"], e["T code"], s, d, t0, t1))

# 2. Only the hosts named in those entries (exact IPs; first three octets for ranges)
exact, ranges = set(), set()
for _, _, _, s, d, _, _ in todo:
    for n in (s, d):
        (exact.add(str(n.network_address)) if n.num_addresses == 1 else ranges.add(prefix(n)))

def keep(c):
    cond = F.col(c).isin(sorted(exact))
    for p in sorted(ranges):
        cond = cond | F.col(c).startswith(p)
    return cond

# 3. Day folders around the entries
days = sorted({(t0 + timedelta(days=k)).strftime("%Y-%m-%d") for *_, t0, t1 in todo for k in (-1, 0, 1)})
jvm, conf = spark._jvm, spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.Path(logs).getFileSystem(conf)
paths = [f"{logs}/{d}/conn.*.log.gz" for d in days
         if fs.globStatus(jvm.org.apache.hadoop.fs.Path(f"{logs}/{d}/conn.*.log.gz"))]
print("Reading", len(paths), "day folders for", len(todo), "entries")

# 4. One pass: connection counts per source, destination and minute (small enough for pandas)
per_minute = (spark.read.json(paths)
              .select(F.col("`id.orig_h`").alias("src"), F.col("`id.resp_h`").alias("dst"),
                      F.coalesce(F.col("ts").cast("double"),
                                 F.to_timestamp("ts").cast("double")).alias("ts"))
              .filter(keep("src") & keep("dst"))
              .withColumn("minute", F.floor(F.col("ts") / 60).cast("long"))
              .groupBy("src", "dst", "minute").count()
              .toPandas())
per_minute["t"] = per_minute["minute"] * 60
print(len(per_minute), "source/destination/minute rows")

# 5. Checks in pandas
def is_in(series, n):
    return series == str(n.network_address) if n.num_addresses == 1 else series.str.startswith(prefix(n))

def total(mask, lo, hi):
    return int(per_minute.loc[mask & per_minute["t"].between(lo - 60, hi), "count"].sum())

out = []
for line, group, tcode, s, d, t0, t1 in todo:
    a, b = t0.timestamp(), t1.timestamp()
    fwd = is_in(per_minute["src"], s) & is_in(per_minute["dst"], d)
    rev = is_in(per_minute["src"], d) & is_in(per_minute["dst"], s)
    r = {"Log line": line, "Group": group, "T code": tcode,
         "fwd logged": total(fwd, a - PAD, b + PAD),
         "fwd +5h":    total(fwd, a + SHIFT - PAD, b + SHIFT + PAD),
         "fwd day":    total(fwd, a - DAY, b + DAY),
         "rev logged": total(rev, a - PAD, b + PAD),
         "rev +5h":    total(rev, a + SHIFT - PAD, b + SHIFT + PAD),
         "rev day":    total(rev, a - DAY, b + DAY)}
    if r["fwd logged"]:   cause = "traffic at the logged time: port or T code does not match the log"
    elif r["fwd +5h"]:    cause = "local time entered as UTC"
    elif r["rev logged"]: cause = "reverse direction (victim -> attacker, e.g. a callback)"
    elif r["rev +5h"]:    cause = "reverse direction and local time entered as UTC"
    elif r["fwd day"]:    cause = "time wrong (traffic the same day)"
    elif r["rev day"]:    cause = "reverse direction and time wrong (same day)"
    else:                 cause = "no traffic between these hosts that day: wrong IP or date, or not captured"
    r["Likely cause"] = cause
    out.append(r)

raw_check = pd.DataFrame(out)
pd.set_option("display.max_rows", None); pd.set_option("display.max_colwidth", None)
display(raw_check["Likely cause"].value_counts().to_frame("entries"))
display(raw_check)
raw_check.to_csv("raw_check_final.csv", index=False)

Reading 31 day folders for 35 entries


[Stage 16:=========================>                              (24 + 8) / 52]

In [ ]:
# Find mission log entries whose time was entered in Central (Kali clock) instead of UTC.
# For every entry missing from the dataset, compare the raw Zeek traffic in two windows:
#   as entered (UTC)   and   converted from Central (America/Chicago) to UTC
# and check whether the traffic in the Central window is on the port the student logged.
# Standalone: needs mission_log_checked_final.csv (from the check cell) in this folder.
import re
import ipaddress
from datetime import timedelta
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from IPython.display import display

spark = (SparkSession.builder.master("local[*]").appName("tz_check")
         .config("spark.driver.memory", "6g").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

logs = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/nsm/zeek/logs"
PAD = 120                                  # the correlation's +/- 2 minutes

def net(text):
    try:
        return ipaddress.ip_network(str(text).strip(), strict=False)
    except ValueError:
        return None

def prefix(n):
    return ".".join(str(n.network_address).split(".")[:3]) + "."

def logged_ports(text):
    """Set of single ports the student entered, or None for a range / 'top 1000' / blank (any port)."""
    t = str(text).split("/")[0].strip()
    if not t or "-" in t or not re.fullmatch(r"[\d,\s]+", t):
        return None
    return {int(p) for p in t.replace(" ", "").split(",") if p}

def central_to_utc(text):
    """Read the entered time as Central (ignore the trailing Z) and convert it to UTC."""
    t = pd.to_datetime(str(text).rstrip("Z"), errors="coerce")
    if pd.isna(t):
        return pd.NaT
    return t.tz_localize("America/Chicago", ambiguous="NaT", nonexistent="NaT").tz_convert("UTC")

# 1. Entries missing from the dataset
results = pd.read_csv("mission_log_checked_final.csv", dtype=str).fillna("")
todo = []
for _, e in results[results["In dataset"] == "No"].iterrows():
    s, d = net(e["Source IP"]), net(e["Dest IP"])
    u0 = pd.to_datetime(e["Start (UTC)"], utc=True, errors="coerce")
    u1 = pd.to_datetime(e["End (UTC)"], utc=True, errors="coerce")
    c0, c1 = central_to_utc(e["Start (UTC)"]), central_to_utc(e["End (UTC)"])
    if s is None or d is None or pd.isna(u0) or pd.isna(u1) or pd.isna(c0) or pd.isna(c1):
        continue
    todo.append({"line": e["Log line"], "group": e["Group"], "tcode": e["T code"],
                 "src": s, "dst": d, "port_text": e["Dest port"], "ports": logged_ports(e["Dest port"]),
                 "u0": min(u0, u1), "u1": max(u0, u1), "c0": min(c0, c1), "c1": max(c0, c1),
                 "entered_start": e["Start (UTC)"], "entered_end": e["End (UTC)"]})

# 2. Hosts and ports named in those entries
exact, ranges, ports = set(), set(), set()
for t in todo:
    for n in (t["src"], t["dst"]):
        (exact.add(str(n.network_address)) if n.num_addresses == 1 else ranges.add(prefix(n)))
    ports |= t["ports"] or set()

def keep(c):
    cond = F.col(c).isin(sorted(exact))
    for p in sorted(ranges):
        cond = cond | F.col(c).startswith(p)
    return cond

days = sorted({(x + timedelta(days=k)).strftime("%Y-%m-%d")
               for t in todo for x in (t["u0"], t["c0"]) for k in (-1, 0, 1)})
jvm, conf = spark._jvm, spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.Path(logs).getFileSystem(conf)
paths = [f"{logs}/{d}/conn.*.log.gz" for d in days
         if fs.globStatus(jvm.org.apache.hadoop.fs.Path(f"{logs}/{d}/conn.*.log.gz"))]
print("Reading", len(paths), "day folders for", len(todo), "entries")

# 3. One pass: counts per source, destination, minute, and port (only logged ports kept; others = -1)
port_list = sorted(ports)
counts = (spark.read.json(paths)
          .select(F.col("`id.orig_h`").alias("src"), F.col("`id.resp_h`").alias("dst"),
                  F.col("`id.resp_p`").cast("int").alias("dport"),
                  F.coalesce(F.col("ts").cast("double"), F.to_timestamp("ts").cast("double")).alias("ts"))
          .filter(keep("src") & keep("dst"))
          .withColumn("port", F.when(F.col("dport").isin(port_list), F.col("dport")).otherwise(F.lit(-1)))
          .withColumn("minute", F.floor(F.col("ts") / 60).cast("long"))
          .groupBy("src", "dst", "port", "minute").count()
          .toPandas())
counts["t"] = counts["minute"] * 60
print(len(counts), "count rows")

# 4. Grade each entry
def is_in(series, n):
    return series == str(n.network_address) if n.num_addresses == 1 else series.str.startswith(prefix(n))

def window(mask, start, end, port_set):
    w = mask & counts["t"].between(start.timestamp() - PAD - 60, end.timestamp() + PAD)
    any_port = int(counts.loc[w, "count"].sum())
    on_port = any_port if port_set is None else int(counts.loc[w & counts["port"].isin(port_set), "count"].sum())
    return any_port, on_port

rows = []
for t in todo:
    fwd = is_in(counts["src"], t["src"]) & is_in(counts["dst"], t["dst"])
    rev = is_in(counts["src"], t["dst"]) & is_in(counts["dst"], t["src"])
    utc_any, utc_port = window(fwd, t["u0"], t["u1"], t["ports"])
    cen_any, cen_port = window(fwd, t["c0"], t["c1"], t["ports"])
    rcen_any, rcen_port = window(rev, t["c0"], t["c1"], t["ports"])

    if utc_port:
        verdict, reverse = "Not a time zone issue (traffic on the logged port at the entered time)", False
    elif cen_port:
        verdict, reverse = "Central time - HIGH (traffic on the logged port after converting)", False
    elif rcen_port:
        verdict, reverse = "Central time + reversed - HIGH (callback on the logged port after converting)", True
    elif cen_any:
        verdict, reverse = "Central time - MEDIUM (traffic between the hosts, but not on the logged port)", False
    elif rcen_any:
        verdict, reverse = "Central time + reversed - MEDIUM (traffic, but not on the logged port)", True
    else:
        verdict, reverse = "Not a time zone issue (no traffic after converting either)", False

    rows.append({"Log line": t["line"], "Group": t["group"], "T code": t["tcode"],
                 "Source IP": str(t["src"]), "Dest IP": str(t["dst"]), "Dest port": t["port_text"],
                 "Entered start": t["entered_start"], "Entered end": t["entered_end"],
                 "Corrected start (UTC)": t["c0"].strftime("%Y-%m-%dT%H:%M:%SZ"),
                 "Corrected end (UTC)": t["c1"].strftime("%Y-%m-%dT%H:%M:%SZ"),
                 "As entered: conns (on port)": f"{utc_any} ({utc_port})",
                 "Central: conns (on port)": f"{cen_any} ({cen_port})",
                 "Central reversed: conns (on port)": f"{rcen_any} ({rcen_port})",
                 "Swap src/dst": "Yes" if reverse else "",
                 "Verdict": verdict})

tz = pd.DataFrame(rows)
pd.set_option("display.max_rows", None); pd.set_option("display.max_colwidth", None)
display(tz["Verdict"].value_counts().to_frame("entries"))
display(tz)
tz.to_csv("timezone_check.csv", index=False)

In [3]:
# Build a corrected copy of the mission log for the correlation rerun.
# Applies only the HIGH-confidence fixes from timezone_check.csv:
#   - times replaced with the Central -> UTC converted times
#   - for callbacks logged attacker -> victim, source and destination IPs swapped
# Every other line is left exactly as it was. Needs timezone_check.csv in this folder.
import pandas as pd
from pyspark.sql import SparkSession
from IPython.display import display

spark = SparkSession.builder.master("local[*]").appName("correct_log").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

mission_log_path = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/attack_log/"
                    "attack_82913-CNT4416-202608-attack.csv")
corrected_file = "attack_82913-CNT4416-202608-attack_corrected.csv"
changes_file = "mission_log_corrections.csv"

# Original log, every column as text, in file order (line 2 = first entry, as in the checks)
log = (spark.read.option("header", True).option("multiline", True).option("encoding", "UTF-8")
       .csv(mission_log_path).toPandas().fillna(""))
original_columns = list(log.columns)
log.index = range(2, 2 + len(log))

tz = pd.read_csv("timezone_check.csv", dtype=str).fillna("")
high = tz[tz["Verdict"].str.contains("HIGH")]

changes = []
for _, h in high.iterrows():
    line = int(h["Log line"])
    row = log.loc[line]
    # Safety check: the entry on this line must be the one that was graded
    if row["mitre_attack_technique"] != h["T code"] or row["datetime_start"] != h["Entered start"]:
        print(f"Line {line}: mission log does not match timezone_check.csv - skipped")
        continue
    change = {"Log line": line, "Group": row["group_id"], "T code": row["mitre_attack_technique"],
              "Old start": row["datetime_start"], "Old end": row["datetime_end"],
              "New start": h["Corrected start (UTC)"], "New end": h["Corrected end (UTC)"],
              "Old source": row["src_ip"], "Old dest": row["dest_ip"],
              "New source": row["src_ip"], "New dest": row["dest_ip"],
              "Reason": "Kali (Central) time converted to UTC"}
    log.loc[line, "datetime_start"] = h["Corrected start (UTC)"]
    log.loc[line, "datetime_end"] = h["Corrected end (UTC)"]
    if h["Swap src/dst"] == "Yes":
        log.loc[line, "src_ip"], log.loc[line, "dest_ip"] = row["dest_ip"], row["src_ip"]
        change["New source"], change["New dest"] = row["dest_ip"], row["src_ip"]
        change["Reason"] += "; callback logged attacker -> victim, IPs swapped"
    changes.append(change)

log[original_columns].to_csv(corrected_file, index=False)
changes = pd.DataFrame(changes)
changes.to_csv(changes_file, index=False)

pd.set_option("display.max_rows", None); pd.set_option("display.max_colwidth", None)
print(f"{len(changes)} entries corrected, {len(log) - len(changes)} left unchanged")
print(f"Corrected log: {corrected_file}   Change list: {changes_file}")
display(changes)

24 entries corrected, 69 left unchanged
Corrected log: attack_82913-CNT4416-202608-attack_corrected.csv   Change list: mission_log_corrections.csv


,Log line,Group,T code,Old start,Old end,New start,New end,Old source,Old dest,New source,New dest,Reason
0,3,2,T1595,2026-09-01T20:56:01Z,2026-09-01T20:56:06Z,2026-09-02T01:56:01Z,2026-09-02T01:56:06Z,143.88.2.13/24,143.88.2.1,143.88.2.13/24,143.88.2.1,Kali (Central) time converted to UTC
1,4,2,T1595,2026-09-01T20:57:01Z,2026-09-01T20:56:02Z,2026-09-02T01:56:02Z,2026-09-02T01:57:01Z,143.88.2.13,143.88.2.10,143.88.2.13,143.88.2.10,Kali (Central) time converted to UTC
2,5,2,T1595,2026-09-01T20:57:30Z,2026-09-01T20:57:35Z,2026-09-02T01:57:30Z,2026-09-02T01:57:35Z,143.88.2.13,143.88.2.11,143.88.2.13,143.88.2.11,Kali (Central) time converted to UTC
3,6,2,T1595,2026-09-01T20:57:38Z,2026-09-01T20:57:40Z,2026-09-02T01:57:38Z,2026-09-02T01:57:40Z,143.88.2.13,143.88.2.12,143.88.2.13,143.88.2.12,Kali (Central) time converted to UTC
4,19,3,T1595,2026-09-05T12:36:08Z,2026-09-05T12:36:15Z,2026-09-05T17:36:08Z,2026-09-05T17:36:15Z,143.88.3.10,143.88.3.12,143.88.3.10,143.88.3.12,Kali (Central) time converted to UTC
5,28,3,T1595,2026-09-09T13:59:00Z,2026-09-09T13:59:10Z,2026-09-09T18:59:00Z,2026-09-09T18:59:10Z,143.88.3.10,143.88.4.11,143.88.3.10,143.88.4.11,Kali (Central) time converted to UTC
6,29,3,T1595,2026-09-09T14:41:00Z,2026-09-09T14:42:20Z,2026-09-09T19:41:00Z,2026-09-09T19:42:20Z,143.88.3.10,143.88.4.12,143.88.3.10,143.88.4.12,Kali (Central) time converted to UTC
7,30,6,T1595,2026-09-09T16:14:00Z,2026-09-09T16:16:00Z,2026-09-09T21:14:00Z,2026-09-09T21:16:00Z,143.88.6.10,143.88.7.11/24,143.88.6.10,143.88.7.11/24,Kali (Central) time converted to UTC
8,35,6,T1595,2026-09-14T14:59:00Z,2026-09-14T15:01:00Z,2026-09-14T19:59:00Z,2026-09-14T20:01:00Z,143.88.6.10,143.88.7.0/24,143.88.6.10,143.88.7.0/24,Kali (Central) time converted to UTC
9,53,10,T1564,2026-09-22T13:50:00Z,2026-09-22T15:05:00Z,2026-09-22T18:50:00Z,2026-09-22T20:05:00Z,143.88.1.11,143.88.10.10,143.88.1.11,143.88.10.10,Kali (Central) time converted to UTC
